## Imports

In [12]:
import os
import json
from abc import ABC, abstractmethod
import chromadb
from typing import List, Dict
from sklearn.metrics import precision_score, recall_score
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from langchain_huggingface import HuggingFacePipeline
import spacy
# import faiss
import tiktoken
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HUGGINGFACE_TOKEN")

## Interfaces

In [13]:
class Chunker(ABC):
    @abstractmethod
    def chunk(self, text: str) -> List[str]:
        ...

class Embedder(ABC):
    @abstractmethod
    def embed(self, texts: List[str]) -> List[List[float]]:
        ...

class VectorStore(ABC):
    @abstractmethod
    def add(self, ids: List[str], texts: List[str], embeddings: List[List[float]]):
        ...

    @abstractmethod
    def query(self, embedding: List[float], top_k: int) -> List[Dict]:
        ...

class Generator(ABC):
    @abstractmethod
    def generate(self, query: str, contexts: List[str]) -> str:
        ...

## Chunking with Overlap using Sliding-Window

In [14]:
class SlidingWindowChunker(Chunker):
    def __init__(self, max_tokens: int = 512, overlap_tokens: int = 256):
        self.tokenizer = tiktoken.get_encoding("cl100k_base")
        self.max_tokens = max_tokens
        self.overlap = overlap_tokens

    def chunk(self, text: str) -> List[str]:
        tokens = self.tokenizer.encode(text)
        chunks = []
        start = 0
        while start < len(tokens):
            end = min(start + self.max_tokens, len(tokens))
            chunk_tokens = tokens[start:end]
            chunks.append(self.tokenizer.decode(chunk_tokens))
            if end == len(tokens):
                break
            start = end - self.overlap
        return chunks

## Generate Embeddings

In [15]:
class TransformerEmbedder(Embedder):
    def __init__(self, model_name: str = "sentence-transformers/all-mpnet-base-v2"):
        self.model = SentenceTransformer(model_name)

    def embed(self, texts: List[str]) -> List[List[float]]:
        # ChromaDB expects embeddings as lists of floats, not torch tensors
        embeddings = self.model.encode(texts, convert_to_tensor=False)
        return embeddings.tolist() if hasattr(embeddings, 'tolist') else embeddings

## ChromaDB Vector Store

In [16]:
class ChromaVectorStore(VectorStore):
    def __init__(self, collection_name: str = "rag_collection", persist_directory: str = "./chroma_db"):
        # Initialize ChromaDB with persistence[12][15]
        self.client = chromadb.PersistentClient(path=persist_directory)
        
        # Create or get collection[13][31]
        try:
            self.collection = self.client.create_collection(name=collection_name)
        except Exception:
            # Collection already exists, get it instead
            self.collection = self.client.get_collection(name=collection_name)

    def add(self, ids: List[str], texts: List[str], embeddings: List[List[float]]):
        """Add documents with embeddings to ChromaDB collection[13][31]"""
        self.collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings
        )

    def query(self, embedding: List[float], top_k: int) -> List[Dict]:
        """Query ChromaDB collection for similar documents[34][37]"""
        results = self.collection.query(
            query_embeddings=[embedding],
            n_results=top_k
        )
        
        # Format results to match expected interface
        formatted_results = []
        if results['documents'] and results['documents'][0]:
            for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
                formatted_results.append({
                    "text": doc,
                    "score": 1 - distance,  # Convert distance to similarity score
                    "id": results['ids'][0][i] if results['ids'] else f"doc_{i}"
                })
        
        return formatted_results


## Llama 3.1 8B Generator

In [17]:
class LlamaGenerator(Generator):
    def __init__(
        self,
        model_id: str = "meta-llama/Meta-Llama-3.1-8B-Instruct",
        device: str = "cuda"
    ):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_length=1024,
            # temperature=0.1,
            do_sample=False,
            eos_token_id=self.tokenizer.eos_token_id
        )

    def generate(self, query: str, contexts: List[str]) -> str:
        ctx = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
        prompt = (
            "You are an expert assistant. Answer the user query using ONLY the contexts below.\n"
            f"User Query: {query}\n\nRelevant Contexts:\n{ctx}\n\n"
            "Answer with citations like “Answer sentence [1][3].”"
        )
        out = self.pipe(prompt)[0]["generated_text"]
        # strip the prompt from the output
        return out[len(prompt):].strip()


## Pipeline

In [18]:
class RAGPipeline:
    def __init__(
        self,
        chunker: Chunker,
        embedder: Embedder,
        store: VectorStore,
        generator: Generator
    ):
        self.chunker = chunker
        self.embedder = embedder
        self.store = store
        self.generator = generator

    def ingest_document(self, doc_id: str, text: str):
        chunks = self.chunker.chunk(text)
        embeddings = self.embedder.embed(chunks)
        ids = [f"{doc_id}-{i}" for i in range(len(chunks))]
        self.store.add(ids, chunks, embeddings)
 
    def query(self, user_query: str, top_k: int = 10) -> str:
        q_emb = self.embedder.embed([user_query])[0]  # Get first embedding from list
        hits = self.store.query(q_emb, top_k)
        contexts = [h["text"] for h in hits]
        return self.generator.generate(user_query, contexts)

## Example Usage

In [19]:
if __name__ == "__main__":
    # Initialize components
    chunker   = SlidingWindowChunker(max_tokens=512, overlap_tokens=64)
    embedder  = TransformerEmbedder()
    store     = ChromaVectorStore(collection_name="my_rag_collection", persist_directory="./chroma_db")
    generator = LlamaGenerator()

    # compose pipeline
    rag = RAGPipeline(chunker, embedder, store, generator)

    # ingest sample document
    with open("sample_doc.txt", encoding="utf-8") as f:
        text = f.read()
    rag.ingest_document("doc1", text)

    # Chunks
    q_emb = rag.embedder.embed(["How do I reset my password?"])[0]
    hits   = rag.store.query(q_emb, top_k=10)           # increase k to 10
    for i,h in enumerate(hits):
        print(f"Rank {i+1}, Score {h['score']:.4f}:\n{h['text']}\n")

    # answer a query
    answer = rag.query("How do I reset my password?")
    print(answer)

c:\Users\Joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
c:\Users\Joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\Joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=Tr

Rank 1, Score -0.6273:
# TechFramework API Documentation

## Overview
TechFramework is a modern web development framework designed for building scalable applications. 
It supports both REST and GraphQL APIs, provides built-in authentication, and includes database integration.

## Authentication
To authenticate with TechFramework API, you need to obtain an API key from the developer console.
Include the API key in your request headers as 'Authorization: Bearer YOUR_API_KEY'.

## Getting Started
1. Install TechFramework: npm install techframework
2. Initialize a new project: techframework init my-project
3. Configure your database connection in config/database.js
4. Start the development server: npm run dev

## Password Reset Functionality
To implement password reset in your application:
1. Create a password reset endpoint: POST /api/auth/reset-password
2. The system will send a reset token to the user's email
3. Users can then use the token to set a new password via PUT /api/auth/update

## Test Queries & Ground Truths

In [23]:
# 1. Define your test queries and ground‐truth contexts/answers
test_queries = [
    "How do I reset my password?",
    "What header should I include for?",
    "How do I start a new TechFramework project?",
    "What is the maximum lifetime of a password reset token?",
    "What command installs TechFramework?"
]

ground_truths = {
    test_queries[0]: [
        "To implement password reset in your application:\n"
        "1. Create a password reset endpoint: POST /api/auth/reset-password\n"
        "2. The system will send a reset token to the user's email\n"
        "3. Users can then use the token to set a new password via PUT /api/auth/update-password\n"
        "4. Tokens expire after 24 hours for security reasons"
    ],
    test_queries[1]: [
        "Include the API key in your request headers as 'Authorization: Bearer YOUR_API_KEY'."
    ],
    test_queries[2]: [
        "To get started:\n"
        "1. Install TechFramework: npm install techframework\n"
        "2. Initialize a new project: techframework init my-project\n"
        "3. Configure your database connection in config/database.js\n"
        "4. Start the development server: npm run dev"
    ],
    test_queries[3]: [
        "Tokens expire after 24 hours for security reasons"
    ],
    test_queries[4]: [
        "Install TechFramework: npm install techframework"
    ]
}

# 2. Run the pipeline to collect retrieved contexts and generated answers
eval_records = []
for q in test_queries:
    # Retrieve and generate
    retrieved = rag.store.query(rag.embedder.embed([q])[0], top_k=5)
    contexts = [r["text"] for r in retrieved]
    answer = rag.generator.generate(q, contexts)
    # Append record
    eval_records.append({
        "user_input": q,
        "retrieved_ids": [r["id"] for r in retrieved],
        "retrieved_texts": contexts,
        "response": answer,
        "ground_truth_contexts": ground_truths[q]
    })


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


## Retrieval Metrics

In [24]:
def precision_at_k(retrieved_ids, ground_truth_ids, k):
    preds = [1 if doc_id in retrieved_ids[:k] else 0 for doc_id in ground_truth_ids]
    # Here ground_truth_ids is the list of really relevant doc IDs
    return precision_score([1]*len(ground_truth_ids), preds, zero_division=0)

def recall_at_k(retrieved_ids, ground_truth_ids, k):
    return len(set(retrieved_ids[:k]) & set(ground_truth_ids)) / len(ground_truth_ids)

# Compute Precision@5 and Recall@5 for each query
for record in eval_records:
    gt_ids = record["ground_truth_contexts"]  # ideally store IDs instead of texts
    rec_ids = record["retrieved_ids"]
    p5 = precision_at_k(rec_ids, gt_ids, 5)
    r5 = recall_at_k(rec_ids, gt_ids, 5)
    print(f"Query: {record['user_input']}\n  Precision@5: {p5:.2f}, Recall@5: {r5:.2f}")


Query: How do I reset my password?
  Precision@5: 0.00, Recall@5: 0.00
Query: What header should I include for?
  Precision@5: 0.00, Recall@5: 0.00
Query: How do I start a new TechFramework project?
  Precision@5: 0.00, Recall@5: 0.00
Query: What is the maximum lifetime of a password reset token?
  Precision@5: 0.00, Recall@5: 0.00
Query: What command installs TechFramework?
  Precision@5: 0.00, Recall@5: 0.00


## Generation Metrics with Ragas

In [25]:
# Convert your records into an EvaluationDataset
eval_dataset = EvaluationDataset.from_list([
    {
      "user_input": rec["user_input"],
      "retrieved_contexts": rec["retrieved_texts"],
      "response": rec["response"],
      "reference": " ".join(rec["ground_truth_contexts"])
    }
    for rec in eval_records
])

# 1. Wrap your Hugging Face pipeline with LangChain's utility
langchain_llm = HuggingFacePipeline(pipeline=rag.generator.pipe)

# 2. Wrap the LangChain-compatible LLM for Ragas[17]
ragas_llm = LangchainLLMWrapper(langchain_llm)

# 3. Run Ragas evaluation
results = evaluate(
    dataset=eval_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=ragas_llm
)

print(results)

Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

c:\Users\Joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\Joo\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1

{'context_recall': nan, 'faithfulness': nan, 'factual_correctness(mode=f1)': nan}
